<a href="https://colab.research.google.com/github/Tagose/Innovative-Proj-Attention_SNN-/blob/main/attention_ssn_hopefully.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Clone da repo
!git clone https://github.com/BICLab/Attention-SNN.git
%cd /content/Attention-SNN
#module structure doesnt has __init__.py. therefore adding it
!touch /content/Attention-SNN/MA_SNN/__init__.py
!touch /content/Attention-SNN/MA_SNN/DVSGestures/__init__.py
!touch /content/Attention-SNN/MA_SNN/DVSGestures/CNN/__init__.py
#numpy.int,float are are not there anymore in ts pytorch ver nowadays
!find /content/Attention-SNN/ -name "*.py" -exec sed -i "s/np.int/int/g" {} +
!find /content/Attention-SNN/ -name "*.py" -exec sed -i "s/np.float/float/g" {} +
#again pytorch doesnt have verbose
target_net = "/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Networks/Att_SNN.py"
!sed -i 's/verbose=config.lr_scheduler_verbose//g' {target_net}
!sed -i 's/, )/)/g' {target_net}
#hard patching gpu batch size because other it runs outta memory
config_path = "/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Config.py"
!sed -i "s/self.device = 'cpu'/self.device = 'cuda'/g" {config_path}
!sed -i "s/self.device_ids = range(0, 0)/self.device_ids = range(0, 1)/g" {config_path}
!sed -i "s/self.batch_size = 128/self.batch_size = 8/g" {config_path}
!sed -i "s/self.batch_size_test = 128/self.batch_size_test = 8/g" {config_path}
#Bypass authors faltu safety checks and change cpu to gpu everywhere
!sed -i 's/torch.device("cuda" if torch.cuda.is_available() else "cpu")/"cuda"/g' /content/Attention-SNN/MA_SNN/DVSGestures/CNN/Att_SNN.py
!sed -i "s/torch.cuda.device_count()/1/g" /content/Attention-SNN/MA_SNN/DVSGestures/CNN/Att_SNN.py

print("Env patched")

Cloning into 'Attention-SNN'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 103 (delta 17), reused 12 (delta 12), pack-reused 73 (from 1)
Receiving objects: 100% (103/103), 260.33 KiB | 11.32 MiB/s, done.
Resolving deltas: 100% (33/33), done.
/content/Attention-SNN
✅ Environment Stabilized and Patched!


In [3]:
import os
%cd /content/Attention-SNN/MA_SNN/DVSGestures/data/

#Download raw dataset
if not os.path.exists('DvsGesture.tar.gz'):
    !wget -O DvsGesture.tar.gz "https://www.dropbox.com/s/cct5kyilhtsliup/DvsGesture.tar.gz?dl=1"
    !tar -xvf DvsGesture.tar.gz

#preprocessing script to create HDF5 data

!python DVS_Gesture.py
print("dun.should work now hopefully")

/content/Attention-SNN/MA_SNN/DVSGestures/data
DVS-Gestures
/content/Attention-SNN/MA_SNN/DVSGestures/data/DVS_Gesture.py:21: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall(path=dirs)
Traceback (most recent call last):
  File "/content/Attention-SNN/MA_SNN/DVSGestures/data/DVS_Gesture.py", line 185, in <module>
    datasets_process(path=path)
  File "/content/Attention-SNN/MA_SNN/DVSGestures/data/DVS_Gesture.py", line 179, in datasets_process
    untar(os.path.join(path, 'DvsGesture.tar.gz'), path)
object address  : 0x7991bf9621a0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C
dun


In [4]:
pathing some files where it coulnd find
network_file = "/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Networks/Att_SNN.py"
config_file = "/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Config.py"

#removing verbose from schdule caller and removing commas and stuff
!sed -i 's/verbose=[A-Za-z]*//g' {network_file}
!sed -i 's/, ,/,/g' {network_file}
!sed -i 's/, )/)/g' {network_file}

#Forcing numwork to  get to only 2 workers
!sed -i "s/self.num_work = 8/self.num_work = 2/g" {config_file}

print("complete")

complete


ok should be ready to launch p much

In [6]:
import sys
import os
import torch

#see if path is still there
project_root = "/content/Attention-SNN/MA_SNN"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

#the main launch part
%cd /content/Attention-SNN/MA_SNN/DVSGestures
from DVSGestures.CNN import Att_SNN

print(f"starting on {torch.cuda.get_device_name(0)}")
Att_SNN.main()

/content/Attention-SNN/MA_SNN/DVSGestures
starting on Tesla T4
cuda
range(0, 1)
dt==25
T==60
attention==no
c_ratio==8
t_ratio==5
epoch==0
num_epochs==300
onlyTest==False
pretrained_path==None
batch_size==8
batch_size_test==8
init_method==None
ds==4
in_channels==2
im_width==32
im_height==32
target_size==11
clip==10
is_train_Enhanced==True
is_spike==False
interval_scaling==False
beta==0
alpha==0.3
Vreset==0
Vthres==0.3
reduction==16
T_extend_Conv==False
T_extend_BN==False
h_conv==False
mem_act==<built-in method relu of type object at 0x7b72f16e4b40>
mode_select==spike
TR_model==NTR
track_running_stats==True
a==0.5
lens==0.25
lr==0.0001
betas==[0.9, 0.999]
eps==1e-08
weight_decay==0
lr_scheduler==True
lr_scheduler_epoch==25
name==no_SNN(CNN)-DVS-Gesture_dt=25ms_T=60
modelPath==/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Result
modelNames==no_SNN(CNN)-DVS-Gesture_dt=25ms_T=60.t7
recordPath==/content/Attention-SNN/MA_SNN/DVSGestures/CNN/Result
recordNames==no_SNN(CNN)-DVS-Gesture_dt=25ms_



  0%|          | 0/147 [00:00<?, ?it/s]

  1%|          | 1/147 [00:01<02:46,  1.14s/it]

Train:Epoch[1/300]:   1%|          | 1/147 [00:01<02:46,  1.14s/it]

Train:Epoch[1/300]:   1%|          | 1/147 [00:01<02:46,  1.14s/it, Loss=0.192]

Train:Epoch[1/300]:   1%|▏         | 2/147 [00:01<02:06,  1.15it/s, Loss=0.192]

Train:Epoch[1/300]:   1%|▏         | 2/147 [00:01<02:06,  1.15it/s, Loss=0.192]

Train:Epoch[1/300]:   1%|▏         | 2/147 [00:02<02:06,  1.15it/s, Loss=0.165]

Train:Epoch[1/300]:   2%|▏         | 3/147 [00:02<01:53,  1.27it/s, Loss=0.165]

Train:Epoch[1/300]:   2%|▏         | 3/147 [00:02<01:53,  1.27it/s, Loss=0.165]

Train:Epoch[1/300]:   2%|▏         | 3/147 [00:02<01:53,  1.27it/s, Loss=0.178]

Train:Epoch[1/300]:   3%|▎         | 4/147 [00:03<01:48,  1.31it/s, Loss=0.178]

Train:Epoch[1/300]:   3%|▎         | 4/147 [00:03<01:48,  1.31it/s, Loss=0.178]

Train:Epoch[1/300]:   3%|▎         | 4/147 [00:03<01:48,  1.31it/s, Loss=0.169]

Train:Epoch[1/300]:   3%|▎   

KeyboardInterrupt: 